# Train D2E2S on Google Colab
Notebook này setup môi trường và train model D2E2S trên dataset `cameraCOQE_quintuple`.

Nếu bạn chưa có code trong Colab, chạy Cell 2 để clone repo.

In [ ]:
# Optional: clone repo if needed
# Thay REPO_URL bằng link repo của bạn nếu chạy notebook mới hoàn toàn trên Colab.
import os
REPO_URL = ""  # vd: 'https://github.com/<user>/DESS-main.git'

if REPO_URL and not os.path.exists('/content/DESS-main'):
    !git clone {REPO_URL} /content/DESS-main

In [ ]:
# Set project root
import os

CANDIDATE_ROOTS = [
    '/content/DESS-main/DESS-main/Codebase',
    '/content/DESS-main/Codebase',
]

PROJECT_ROOT = None
for p in CANDIDATE_ROOTS:
    if os.path.exists(p):
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Khong tim thay Codebase. Hay clone repo hoac sua PROJECT_ROOT.')

%cd {PROJECT_ROOT}
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip -q install -U pip
!pip -q install -r requirements.txt
# torch-geometric is required by models/TIN_GCN.py
!pip -q install torch-geometric

In [ ]:
# Quick sanity check for dataset files
import os
required = [
    'data/cameraCOQE_quintuple/train_quintuple.json',
    'data/cameraCOQE_quintuple/dev_quintuple.json',
    'data/cameraCOQE_quintuple/test_quintuple.json',
    'data/types.json',
]
for f in required:
    print(f, 'OK' if os.path.exists(f) else 'MISSING')

## Train command
Bạn có thể chỉnh các tham số bên dưới trước khi chạy.
- `eval_match_mode=both`: in cả index-match và span-match
- `sen_filter_threshold`, `sentence_filter_threshold`: ngưỡng lọc khi eval

In [ ]:
import os
import subprocess
import shlex

if PROJECT_ROOT is None:
    raise ValueError('PROJECT_ROOT is not set. Run Cell 3 first.')

# Optional: write logs/checkpoints to Google Drive
OUTPUT_ROOT = PROJECT_ROOT  # vd: '/content/drive/MyDrive/dess_runs/run1'

cmd = [
    'python', os.path.join(PROJECT_ROOT, 'train.py'),
    '--dataset', 'cameraCOQE_quintuple',
    '--pretrained_deberta_name', 'microsoft/deberta-v3-base',
    '--emb_dim', '768',
    '--deberta_feature_dim', '768',
    '--hidden_dim', '384',
    '--batch_size', '4',
    '--epochs', '20',
    '--lr', '1e-5',
    '--sampling_processes', '0',
    '--max_role_candidates', '4',
    '--max_pairs', '1000',
    '--sen_filter_threshold', '0.70',
    '--sentence_filter_threshold', '0.60',
    '--sentence_loss_weight', '0.2',
    '--neg_entity_count', '25',
    '--neg_triple_count', '60',
    '--eval_match_mode', 'both',
    '--log_path', os.path.join(OUTPUT_ROOT, 'log'),
    '--save_path', os.path.join(OUTPUT_ROOT, 'savemodels'),
]

print('Running command:\n' + ' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)

In [ ]:
# View latest result file
import glob
files = sorted(glob.glob('log/*/result*.txt') + glob.glob('log/result*.txt'))
print('Found result files:', files[-3:])
if files:
    print('\n===== Tail of latest result file =====')
    !tail -n 120 {files[-1]}